# Deep Learning QSAR Models — LSTM, GNN, CGCNN

Train deep learning models on molecular representations and compare with traditional QSAR baselines.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load curated data

In [ ]:
df = pd.read_csv('data/curated_data_rdkit.csv')
print(f'Dataset: {df.shape[0]} molecules, {df.shape[1]} columns')
print(f'Target: log_LC50 range [{df["log_LC50"].min():.2f}, {df["log_LC50"].max():.2f}]')
df.head()

## 2. Prepare molecular graphs for GNN/CGCNN

In [ ]:
ATOM_FEATURES = {
    'C': 0, 'N': 1, 'O': 2, 'S': 3, 'P': 4, 'F': 5, 'Cl': 6, 'Br': 7, 'I': 8,
    'B': 9, 'Si': 10, 'Se': 11, 'Te': 12, 'Sn': 13, 'As': 14, 'Sb': 15,
    'H': 16, 'default': 17
}

BOND_TYPES = {Chem.BondType.SINGLE: 0, Chem.BondType.DOUBLE: 1, Chem.BondType.TRIPLE: 2, Chem.BondType.AROMATIC: 3}

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    atom_symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    x = torch.tensor([[ATOM_FEATURES.get(s, ATOM_FEATURES['default'])] for s in atom_symbols], dtype=torch.long)
    
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = BOND_TYPES.get(bond.GetBondType(), 0)
        edge_index.extend([[i, j], [j, i]])
        edge_attr.extend([bt, bt])
    
    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0,), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).T
        edge_attr = torch.tensor(edge_attr, dtype=torch.long)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

In [ ]:
smiles_list = df['SMILES'].tolist()
graphs = [mol_to_graph(s) for s in smiles_list]
valid_mask = [g is not None for g in graphs]
graphs = [g for g, v in zip(graphs, valid_mask) if v]
y_all = torch.tensor(df.loc[valid_mask, 'log_LC50'].values, dtype=torch.float32)
print(f'Valid graphs: {len(graphs)} / {len(smiles_list)}')

## 3. Prepare fingerprint features for LSTM

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mols = [Chem.MolFromSmiles(s) for s in df.loc[valid_mask, 'SMILES'].tolist()]
fps = np.array([gen.GetFingerprint(m) for m in mols])
print(f'Fingerprint matrix: {fps.shape}')

## 4. Train/test split

In [ ]:
indices = np.arange(len(graphs))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_graphs = [graphs[i] for i in train_idx]
test_graphs = [graphs[i] for i in test_idx]
y_train = y_all[train_idx]
y_test = y_all[test_idx]

fps_train = fps[train_idx]
fps_test = fps[test_idx]

scaler = StandardScaler()
fps_train_scaled = scaler.fit_transform(fps_train)
fps_test_scaled = scaler.transform(fps_test)

print(f'Train: {len(train_graphs)}, Test: {len(test_graphs)}')

## 5. LSTM Model

In [ ]:
class FingerprintDataset(Dataset):
    def __init__(self, fps, targets):
        self.fps = torch.tensor(fps, dtype=torch.float32)
        self.targets = targets
    
    def __len__(self):
        return len(self.fps)
    
    def __getitem__(self, idx):
        return self.fps[idx], self.targets[idx]

class LSTMRegressor(nn.Module):
    def __init__(self, input_size=2048, hidden_size=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, 
                           num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        return self.fc(out).squeeze(-1)

In [ ]:
def train_model(model, train_loader, val_data, epochs=100, lr=1e-3, patience=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(batch_y)
        
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        with torch.no_grad():
            val_x, val_y = val_data
            val_pred = model(val_x.to(device))
            val_loss = criterion(val_pred, val_y.to(device)).item()
        
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

In [ ]:
train_dataset = FingerprintDataset(fps_train_scaled, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_data = (torch.tensor(fps_test_scaled, dtype=torch.float32), y_test)

lstm_model = LSTMRegressor(input_size=2048, hidden_size=256, num_layers=2, dropout=0.3).to(device)
lstm_model, lstm_history = train_model(lstm_model, train_loader, val_data, epochs=150, lr=1e-3, patience=20)

In [ ]:
lstm_model.eval()
with torch.no_grad():
    lstm_pred = lstm_model(val_data[0].to(device)).cpu().numpy()

lstm_r2 = r2_score(y_test.numpy(), lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test.numpy(), lstm_pred))
lstm_mae = mean_absolute_error(y_test.numpy(), lstm_pred)
print(f'LSTM — R²={lstm_r2:.4f}, RMSE={lstm_rmse:.4f}, MAE={lstm_mae:.4f}')

## 6. GNN Model (Graph Convolutional Network)

In [ ]:
class GraphDataset(Dataset):
    def __init__(self, graphs, targets):
        self.graphs = graphs
        self.targets = targets
    
    def __len__(self):
        return len(self.graphs)
    
    def __getitem__(self, idx):
        return self.graphs[idx], self.targets[idx]

def collate_graphs(batch):
    graphs, targets = zip(*batch)
    batch = Batch.from_data_list(list(graphs))
    targets = torch.tensor(targets, dtype=torch.float32)
    return batch, targets

class GCNRegressor(nn.Module):
    def __init__(self, num_node_features=18, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(num_node_features, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, data):
        x, edge_index, batch = data.x.squeeze(-1), data.edge_index, data.batch
        x = self.embedding(x)
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze(-1)

In [ ]:
train_graph_dataset = GraphDataset(train_graphs, y_train)
train_graph_loader = DataLoader(train_graph_dataset, batch_size=32, shuffle=True, collate_fn=collate_graphs)

test_graph_dataset = GraphDataset(test_graphs, y_test)
test_graph_loader = DataLoader(test_graph_dataset, batch_size=32, shuffle=False, collate_fn=collate_graphs)

gcn_model = GCNRegressor(num_node_features=18, hidden_dim=128, dropout=0.3).to(device)

def train_gnn(model, train_loader, test_loader, epochs=150, lr=1e-3, patience=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_data, batch_y in train_loader:
            batch_data, batch_y = batch_data.to(device), batch_y.to(device)
            optimizer.zero_grad()
            pred = model(batch_data)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(batch_y)
        
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_data, batch_y in test_loader:
                batch_data, batch_y = batch_data.to(device), batch_y.to(device)
                pred = model(batch_data)
                val_loss += criterion(pred, batch_y).item() * len(batch_y)
        val_loss /= len(test_loader.dataset)
        
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

gcn_model, gcn_history = train_gnn(gcn_model, train_graph_loader, test_graph_loader, epochs=150, lr=1e-3, patience=20)

In [ ]:
gcn_model.eval()
gcn_preds = []
gcn_targets = []
with torch.no_grad():
    for batch_data, batch_y in test_graph_loader:
        batch_data = batch_data.to(device)
        pred = gcn_model(batch_data)
        gcn_preds.extend(pred.cpu().numpy())
        gcn_targets.extend(batch_y.numpy())

gcn_r2 = r2_score(gcn_targets, gcn_preds)
gcn_rmse = np.sqrt(mean_squared_error(gcn_targets, gcn_preds))
gcn_mae = mean_absolute_error(gcn_targets, gcn_preds)
print(f'GCN — R²={gcn_r2:.4f}, RMSE={gcn_rmse:.4f}, MAE={gcn_mae:.4f}')

## 7. CGCNN Model (Crystal Graph Convolutional Neural Network adapted for molecules)

In [ ]:
class CGCNNConv(nn.Module):
    def __init__(self, node_dim=64, edge_dim=4):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(node_dim * 2 + edge_dim, node_dim),
            nn.Softplus()
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim, node_dim),
            nn.Sigmoid()
        )
        self.transform = nn.Linear(node_dim, node_dim)
    
    def forward(self, x, edge_index, edge_attr):
        row, col = edge_index
        edge_features = torch.cat([x[row], x[col], edge_attr], dim=-1)
        m = self.edge_mlp(edge_features)
        
        out = torch.zeros_like(x)
        out.index_add_(0, col, m)
        deg = torch.zeros(x.size(0), device=x.device)
        deg.index_add_(0, col, torch.ones(m.size(0), device=x.device))
        deg = deg.clamp(min=1).unsqueeze(-1)
        m_agg = out / deg
        
        z = self.node_mlp(m_agg)
        x_new = x + z * self.transform(m_agg)
        return x_new

class CGCNNRegressor(nn.Module):
    def __init__(self, num_node_features=18, edge_dim=4, node_dim=64, num_conv=3, dropout=0.2):
        super().__init__()
        self.node_embedding = nn.Embedding(num_node_features, node_dim)
        self.edge_embedding = nn.Embedding(edge_dim, edge_dim)
        
        self.convs = nn.ModuleList([CGCNNConv(node_dim, edge_dim) for _ in range(num_conv)])
        
        self.fc = nn.Sequential(
            nn.Linear(node_dim, 64),
            nn.Softplus(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.Softplus(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, data):
        x = self.node_embedding(data.x.squeeze(-1))
        edge_attr = self.edge_embedding(data.edge_attr)
        edge_index = data.edge_index
        
        for conv in self.convs:
            x = conv(x, edge_index, edge_attr)
        
        x = global_mean_pool(x, data.batch)
        return self.fc(x).squeeze(-1)

In [ ]:
cgcnn_model = CGCNNRegressor(num_node_features=18, edge_dim=4, node_dim=64, num_conv=3, dropout=0.2).to(device)
cgcnn_model, cgcnn_history = train_gnn(cgcnn_model, train_graph_loader, test_graph_loader, epochs=150, lr=1e-3, patience=20)

In [ ]:
cgcnn_model.eval()
cgcnn_preds = []
cgcnn_targets = []
with torch.no_grad():
    for batch_data, batch_y in test_graph_loader:
        batch_data = batch_data.to(device)
        pred = cgcnn_model(batch_data)
        cgcnn_preds.extend(pred.cpu().numpy())
        cgcnn_targets.extend(batch_y.numpy())

cgcnn_r2 = r2_score(cgcnn_targets, cgcnn_preds)
cgcnn_rmse = np.sqrt(mean_squared_error(cgcnn_targets, cgcnn_preds))
cgcnn_mae = mean_absolute_error(cgcnn_targets, cgcnn_preds)
print(f'CGCNN — R²={cgcnn_r2:.4f}, RMSE={cgcnn_rmse:.4f}, MAE={cgcnn_mae:.4f}')

## 8. Comparison with traditional QSAR models

In [ ]:
traditional_results = {
    'XGBoost': {'R2': 0.5093, 'RMSE': 0.9180, 'MAE': 0.6422},
    'LightGBM': {'R2': 0.5078, 'RMSE': 0.9194, 'MAE': 0.6507},
    'RandomForest': {'R2': 0.4730, 'RMSE': 0.9514, 'MAE': 0.6689},
}

dl_results = {
    'LSTM': {'R2': lstm_r2, 'RMSE': lstm_rmse, 'MAE': lstm_mae},
    'GCN': {'R2': gcn_r2, 'RMSE': gcn_rmse, 'MAE': gcn_mae},
    'CGCNN': {'R2': cgcnn_r2, 'RMSE': cgcnn_rmse, 'MAE': cgcnn_mae},
}

all_results = {**traditional_results, **dl_results}
comparison_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
comparison_df.index.name = 'Model'
print(comparison_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_list = list(comparison_df.index)
x = np.arange(len(models_list))
width = 0.25

colors_traditional = ['#2196F3', '#1976D2', '#0D47A1']
colors_dl = ['#FF5722', '#E64A19', '#D84315']
colors = colors_traditional + colors_dl

axes[0].bar(x, comparison_df['R2'], color=colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_list, rotation=45, ha='right')
axes[0].set_ylabel('R²')
axes[0].set_title('R² Score')

axes[1].bar(x, comparison_df['RMSE'], color=colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_list, rotation=45, ha='right')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE')

axes[2].bar(x, comparison_df['MAE'], color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels(models_list, rotation=45, ha='right')
axes[2].set_ylabel('MAE')
axes[2].set_title('MAE')

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/dl_vs_traditional_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Parity plots for deep learning models

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

dl_preds = {'LSTM': lstm_pred, 'GCN': gcn_preds, 'CGCNN': cgcnn_preds}
dl_metrics = {'LSTM': (lstm_r2, lstm_rmse), 'GCN': (gcn_r2, gcn_rmse), 'CGCNN': (cgcnn_r2, cgcnn_rmse)}

for ax, (name, preds) in zip(axes, dl_preds.items()):
    r2, rmse = dl_metrics[name]
    ax.scatter(y_test.numpy(), np.array(preds).flatten(), alpha=0.5, edgecolors='none')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    ax.set_xlabel('Actual log_LC50')
    ax.set_ylabel('Predicted log_LC50')
    ax.set_title(f'{name} (R²={r2:.3f}, RMSE={rmse:.3f})')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('figures/dl_parity_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save models

In [ ]:
os.makedirs('models', exist_ok=True)

torch.save(lstm_model.state_dict(), 'models/LSTM_trained.pth')
torch.save(gcn_model.state_dict(), 'models/GCN_trained.pth')
torch.save(cgcnn_model.state_dict(), 'models/CGCNN_trained.pth')
joblib.dump(scaler, 'models/LSTM_scaler.pkl')

print('Models saved to models/')
print(f'  LSTM_trained.pth, GCN_trained.pth, CGCNN_trained.pth, LSTM_scaler.pkl')